# Generator reactive-power (Q) limits

DPsim's Newton-Raphson powerflow solver can enforce generator Qmin/Qmax limits with a bidirectional PV<->PQ outer loop: a PV bus whose generator reactive output exceeds its limit is pinned at the limit and the bus is converted to PQ; if the constraint later relaxes, the bus reverts to PV. Enforcement is opt-in via `set_pf_solver_enforce_q_limits(True)` on the `Simulation` (default off, so a case with no limits set behaves exactly as before).

This notebook uses a small hand-wired three-bus case (slack, a PiLine, a PV generator with configurable Qmax/Qmin, another PiLine, a PQ load) and sweeps a set of cases designed to exercise the limit comparison, not just its most common shape:

- a tight, moderate, and loose Qmax binding the generator's natural (lagging-load) output
- a window that does not bind at all
- both limits positive (a generator required to always produce Q, binding on the lower bound)
- both limits negative (a generator required to always absorb Q, binding on the upper bound)
- a capacitive (leading) load, giving the generator a negative natural Q, so a genuinely negative Qmin binds from a negative starting point

Each case is run under both the dense and sparse Jacobian solvers and must agree.

In [ ]:
import math
import numpy as np
import pandas as pd
import dpsimpy

In [ ]:
BASE_V = 230e3
BASE_S = 100e6
V_SET = 1.02


def run_pf(q_max_mvar, q_min_mvar, load_q_mvar, enforce, use_sparse):
    n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single)
    n2 = dpsimpy.sp.SimNode("n2", dpsimpy.PhaseType.Single)
    n3 = dpsimpy.sp.SimNode("n3", dpsimpy.PhaseType.Single)

    slack = dpsimpy.sp.ph1.SynchronGenerator("slack")
    slack.set_parameters(
        rated_apparent_power=BASE_S,
        rated_voltage=BASE_V,
        set_point_active_power=0.0,
        set_point_voltage=BASE_V,
        powerflow_bus_type=dpsimpy.PowerflowBusType.VD,
    )
    slack.set_base_voltage(BASE_V)
    slack.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.VD)
    slack.connect([n1])

    gen = dpsimpy.sp.ph1.SynchronGenerator("gen")
    gen.set_parameters(
        rated_apparent_power=BASE_S,
        rated_voltage=BASE_V,
        set_point_active_power=50e6,
        set_point_voltage=V_SET * BASE_V,
        powerflow_bus_type=dpsimpy.PowerflowBusType.PV,
        q_limit_max=q_max_mvar * 1e6,
        q_limit_min=q_min_mvar * 1e6,
    )
    gen.set_base_voltage(BASE_V)
    gen.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PV)
    gen.connect([n2])

    load = dpsimpy.sp.ph1.Load("load")
    load.set_parameters(120e6, load_q_mvar * 1e6, BASE_V)
    load.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PQ)
    load.connect([n3])

    z_base = BASE_V**2 / BASE_S
    line12 = dpsimpy.sp.ph1.PiLine("line12")
    line12.set_parameters(0.02 * z_base, 0.06 * z_base / (2 * math.pi * 50), 0.0, 0.0)
    line12.set_base_voltage(BASE_V)
    line12.connect([n1, n2])

    line23 = dpsimpy.sp.ph1.PiLine("line23")
    line23.set_parameters(0.02 * z_base, 0.06 * z_base / (2 * math.pi * 50), 0.0, 0.0)
    line23.set_base_voltage(BASE_V)
    line23.connect([n2, n3])

    system = dpsimpy.SystemTopology(
        50, [n1, n2, n3], [slack, gen, load, line12, line23]
    )

    sim = dpsimpy.Simulation("qlimit_pf", dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_time_step(0.1)
    sim.set_final_time(0.1)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Simulation)
    sim.set_pf_solver_use_sparse(use_sparse)
    sim.set_pf_solver_enforce_q_limits(enforce)
    sim.run()

    s_gen = complex(np.asarray(n2.attr("s").get()).flatten()[0])
    v_gen = abs(complex(n2.single_voltage()) / BASE_V)
    return v_gen, s_gen.imag / 1e6

With the default lagging load ($Q_{load}=120$ MVAr) the generator's natural, unconstrained reactive output is about +203 MVAr. With a leading (capacitive) load ($Q_{load}=-160$ MVAr) it is about -77 MVAr. The case table below picks Qmax/Qmin around both starting points so every comparison branch in the enforcement logic (exceeds Qmax, falls below Qmin, neither) is exercised at least once, including with same-sign and negative-starting-point limits.

In [ ]:
# label, Qmax [MVAr], Qmin [MVAr], load Q [MVAr], expected outcome
CASES = [
    ("tight Qmax", 5, -200, 120, "max"),
    ("moderate Qmax", 50, -200, 120, "max"),
    ("loose Qmax, still binding", 150, -200, 120, "max"),
    ("asymmetric window, non-binding", 300, -200, 120, "hold"),
    ("both limits positive, binds Qmin", 300, 250, 120, "min"),
    ("both limits negative, binds Qmax", -20, -150, 120, "max"),
    ("leading load, binds Qmin from negative start", 150, -30, -160, "min"),
    ("leading load, non-binding window", 150, -200, -160, "hold"),
]

In [ ]:
rows = []
for use_sparse in (False, True):
    for label, q_max, q_min, load_q, expected in CASES:
        v, q = run_pf(q_max, q_min, load_q, enforce=True, use_sparse=use_sparse)
        rows.append(
            {
                "case": label,
                "solver": "sparse" if use_sparse else "dense",
                "Qmax [MVAr]": q_max,
                "Qmin [MVAr]": q_min,
                "expected": expected,
                "V [pu]": v,
                "gen Q [MVAr]": q,
            }
        )
results = pd.DataFrame(rows)
results

In [ ]:
for _, row in results.iterrows():
    if row["expected"] == "max":
        assert (
            abs(row["gen Q [MVAr]"] - row["Qmax [MVAr]"]) < 0.5
        ), f"{row['solver']}/{row['case']}: not pinned at Qmax"
    elif row["expected"] == "min":
        assert (
            abs(row["gen Q [MVAr]"] - row["Qmin [MVAr]"]) < 0.5
        ), f"{row['solver']}/{row['case']}: not pinned at Qmin"
    else:
        assert (
            abs(row["V [pu]"] - V_SET) < 1e-3
        ), f"{row['solver']}/{row['case']}: PV bus not held at V_set"

# enforcement-off must be untouched by the feature, both load directions
v_lag, q_lag = run_pf(1e9, -1e9, 120, enforce=False, use_sparse=False)
v_lead, q_lead = run_pf(1e9, -1e9, -160, enforce=False, use_sparse=False)
assert abs(v_lag - V_SET) < 1e-3 and abs(v_lead - V_SET) < 1e-3

# dense and sparse solvers must agree on every case
dense = results[results["solver"] == "dense"].reset_index(drop=True)
sparse = results[results["solver"] == "sparse"].reset_index(drop=True)
assert np.allclose(dense["V [pu]"], sparse["V [pu]"], atol=1e-8)
assert np.allclose(dense["gen Q [MVAr]"], sparse["gen Q [MVAr]"], atol=1e-6)

print(f"unlimited gen Q: lagging load {q_lag:.2f} MVAr, leading load {q_lead:.2f} MVAr")
print("Q-limit enforcement verified across sign/asymmetry cases, dense == sparse.")